### Middlewares

In [2]:

import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


### Summarization Middleware

Summarization Middleware in Agentic AI is a middleware component that automatically compresses long conversations, documents, or execution histories into concise summaries so that the agent can retain important context without exceeding token limits.

In [4]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage

### MessageBased Summarization
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b", ## this model will be used to summarise the messages
            trigger=("messages",10), ## Will Summarise the messages when the count of messages reaches to 10
            keep=("messages",4) ## Will keep last 4 messages as it is and replace the 1 - 6 messages with the summarised message
        )
    ]
)

In [5]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

In [6]:
questions=[
    "What is 2+3",
    "What is 2+4",
    "What is 2+5",
    "What is 2+6",
    "What is 2+7",
    "What is 2+8",
]

In [7]:
for q in questions:
    response=agent.invoke({"messages":[HumanMessage(content=q)]},config)
    print(f"Messages: {response}")
    print(f"Messages: {len(response["messages"])}")

Messages: {'messages': [HumanMessage(content='What is 2+3', additional_kwargs={}, response_metadata={}, id='89a0ef9a-150e-403f-ab3d-fcb33d1d7918'), AIMessage(content='<think>\nOkay, let\'s see. The user is asking "What is 2+3". Hmm, that\'s a basic arithmetic question. I need to make sure I understand it correctly. So, addition of two numbers, 2 and 3. Let me break it down.\n\nFirst, 2 plus 3. Well, in arithmetic, when you add 2 and 3 together, you get 5. But wait, maybe they want a more detailed explanation? Let me think. If someone is asking this, they might be a beginner learning basic math. So, perhaps I should explain it step by step.\n\nStarting with the number 2, if you add 3 to it, you can count forward three times: 2, then 3 (adding 1), 4 (adding another 1), and 5 (adding the last 1). So, 2 + 3 equals 5. Alternatively, using objects: if you have two apples and get three more, you now have five apples. That\'s another way to visualize it.\n\nIs there any chance of a trick quest

### Token Size

In [2]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langgraph.checkpoint.memory import InMemorySaver
from langchain_core.messages import HumanMessage,SystemMessage
from langchain_core.tools import tool

In [3]:
@tool
def search_hotels(city:str)->str:
    """Search hotels - returns long response to use more tokens."""
    return f"""Hotels in {city}:
    1. Grand Hotel
    2. City Inn
    3. Budget Stay"""


In [4]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    checkpointer=InMemorySaver(),
    tools=[search_hotels],
    middleware=[
        SummarizationMiddleware(
            model="groq:qwen/qwen3-32b", 
            trigger=("tokens",550), 
            keep=("tokens",200) 
        )
    ]
)

In [5]:
### Run with thread id
config={"configurable":{"thread_id":"test-1"}}

def count_tokens(messages):
    total_chars=sum(len(str(m.content)) for m in messages)
    return total_chars // 4

In [8]:
citites=["Paris","Mumbai","New York","London","Tokyo"]

for city in citites:
    response=agent.invoke(
        {"messages":[HumanMessage(content=f"Find hotels in {city}")]},
        config=config
    )

    tokens=count_tokens(response["messages"])
    print(f"{city}: ~{tokens} tokens, {len(response["messages"])} messages")
    print(f"{(response["messages"])}")

Paris: ~1214 tokens, 6 messages
[HumanMessage(content='Here is a summary of the conversation to date:\n\n<think>\nOkay, let\'s see. The user wants me to extract the most relevant context from their conversation history. The main goal here is to free up space by replacing the history with a concise summary.\n\nFirst, looking at the messages: the user asked for hotels in Paris. The AI called a tool with the city set to Paris. The tool responded with three hotel names. \n\nFor the SESSION INTENT, the user\'s primary goal is clearly to find hotels in Paris. That\'s straightforward.\n\nIn the SUMMARY, I need to capture the key info: the user\'s request, the tool used, the results provided. Also note that there\'s no further interaction, so maybe mention that the user hasn\'t selected a hotel yet. But the instructions say to include important choices and reasoning. Since there are no choices made yet, just the search, the summary should reflect the action taken and the response received.\n\n

RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for model `qwen/qwen3-32b` in organization `org_01kpmm0ydye6pt5bm91p1n5w6y` service tier `on_demand` on tokens per minute (TPM): Limit 6000, Used 4553, Requested 1714. Please try again in 2.67s. Need more tokens? Upgrade to Dev Tier today at https://console.groq.com/settings/billing', 'type': 'tokens', 'code': 'rate_limit_exceeded'}}

### Human in the loop middleware

In [30]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import InMemorySaver

In [31]:
def read_email_tool(email_id:str)->str:
    """Mock function to read an email by its ID."""
    return f"Email content for ID: {email_id}"

In [32]:
def send_email_tool(recipient:str, subject:str, body:str)->str:
    """Mock the function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}"

In [33]:
agent=create_agent(
    model="groq:qwen/qwen3-32b",
    tools=[read_email_tool,send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve","edit","reject"]
                },
                "read_email_tool":False
            }
        )
    ]
)

In [34]:
config={"configurable":{"thread_id":"test-approve"}}

result=agent.invoke(
    {"messages":[HumanMessage(content="Send an email to jon@test.com with subject 'Hello' and body 'How are you?'")]},
    config=config
)

In [35]:
result

{'messages': [HumanMessage(content="Send an email to jon@test.com with subject 'Hello' and body 'How are you?'", additional_kwargs={}, response_metadata={}, id='310b54c9-7146-4755-a2fd-97ed89710109'),
  AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user wants to send an email to jon@test.com with the subject 'Hello' and body 'How are you?'. Let me check the available tools. There's a send_email_tool that requires recipient, subject, and body. All three are required. The parameters are all strings. The user provided all three pieces of information, so I can use this tool. I need to structure the arguments correctly. The recipient is jon@test.com, subject is Hello, body is How are you? I'll make sure to format the JSON properly without any syntax errors. Let me double-check the parameter names and types. Yep, looks good. So the function call should include all three parameters as specified.\n", 'tool_calls': [{'id': 't9ys6fw11', 'function': {'arguments': '{"bod

In [36]:
from langgraph.types import Command

In [ ]:
if "__interrupt__" in result:
    print("Paused! Approving...")

    result=agent.invoke(
        Command(
            resume={
                "decisions":[
                    {"type":"reject"} ## The AI will reject the action
                ]
            }
        ),
        config=config
    )

    print(f"Result: {result['messages'][-1].content}")

Paused! Approving...
Result: It seems the email request was rejected. Would you like me to:

1. Attempt to send the email again?
2. Check if there's an issue with the email content?
3. Help modify the message before resending?

Please let me know how you'd like to proceed.
